In [11]:
#!/usr/bin/env python3
"""
Jupyter version of:
Filter matched_all_2024_methods.csv by AWE count ratio and remove troublesome orbits.

Defaults:
  - input: results/matched_saber_awe_600_s_500_km_full_cross/matched_all_2024_2025_methods.csv
  - threshold: 0.90
  - drop_orbits: ["03672", "03673", "04172", "05575", "06190", "6393"]
  - drop_awe_substrings: []
  - output: auto (adds suffix like _ratio90 or _ratio90_clean)

Usage in a notebook:
    df_out, out_path = filter_matched_all()   # just run with defaults
    # or customize a bit:
    # df_out, out_path = filter_matched_all(threshold=0.80, drop_orbits=["03672","03673"])
"""

from pathlib import Path
import pandas as pd
import numpy as np
import re
from typing import Optional, Iterable, Tuple

# ---------------------------
# Helpers (unchanged logic)
# ---------------------------
ORBIT_RES = [
    re.compile(r'\bO?(\d{5})\b'),            # 03672   or O03672
    re.compile(r'[^\d](\d{5})[^\d]'),        # ..._03672_...
    re.compile(r'Orbit[_-]?(\d{5})', re.I),  # Orbit_03672 or Orbit-03672
]

def extract_orbit_from_name(name: str) -> Optional[str]:
    if not isinstance(name, str):
        return None
    s = f"_{name}_"  # pad to help regex with boundaries
    for rx in ORBIT_RES:
        m = rx.search(s)
        if m:
            return m.group(1)
    return None

def _build_output_path(in_path: Path, threshold: float, has_clean_tag: bool) -> Path:
    percent = int(round(threshold * 100))
    suffix = f"_ratio{percent}"
    if has_clean_tag:
        suffix += "_clean"
    return in_path.with_name(in_path.stem + suffix + ".csv")

# ---------------------------------
# Notebook-friendly main function
# ---------------------------------
def filter_matched_all(
    input_path: str | Path = "matched_all_2024_2025_methods.csv",
    threshold: float = 0.90,
    drop_orbits: Iterable[str] = ("03672", "03673", "04172", "05575", "06190", "6393"),
    drop_awe_substrings: Iterable[str] = (),
    output_path: str | Path | None = None,
    write_csv: bool = True,
) -> Tuple[pd.DataFrame, Optional[Path]]:
    """
    Returns (filtered_dataframe, out_path_or_None).
    Also prints a short report. Set write_csv=False to skip saving.
    """
    in_path = Path(input_path)
    if not in_path.exists():
        raise FileNotFoundError(in_path)

    df = pd.read_csv(in_path)

    # ----- ensure columns -----
    needed = {"awe_count_used", "awe_count_unfiltered", "awe_file_nearest"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # ----- ratio filter -----
    denom = df["awe_count_unfiltered"].replace(0, np.nan)
    ratio = df["awe_count_used"] / denom
    df["awe_ratio_used_over_unfiltered"] = ratio
    mask_ratio_ok = ratio.ge(threshold) & ratio.replace([np.inf, -np.inf], np.nan).notna() & (df["awe_count_unfiltered"] > 0)

    # ----- build drop masks -----
    bad_orbits = set(str(x) for x in (drop_orbits or ()))
    bad_subs = set(drop_awe_substrings or ())

    # Extract orbit field from AWE filename for easy filtering
    df["awe_orbit"] = df["awe_file_nearest"].apply(extract_orbit_from_name)

    drop_mask = pd.Series(False, index=df.index)
    if bad_orbits:
        drop_mask |= df["awe_orbit"].isin(bad_orbits)

    if bad_subs:
        pat = "|".join(re.escape(s) for s in bad_subs)
        drop_mask |= df["awe_file_nearest"].astype(str).str.contains(pat, case=False, na=False)

    # Combine masks
    final_mask = mask_ratio_ok & ~drop_mask
    df_out = df.loc[final_mask].copy()

    # ----- save (optional) -----
    out_path = None
    if write_csv:
        if output_path is None:
            out_path = _build_output_path(in_path, threshold=threshold, has_clean_tag=bool(bad_orbits or bad_subs))
        else:
            out_path = Path(output_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df_out.to_csv(out_path, index=False)

    # ----- report -----
    print(f"Input rows: {len(df)}")
    print(f"Kept after ratio >= {threshold:.2f}: {int(mask_ratio_ok.sum())}")
    if bad_orbits or bad_subs:
        print(f"Dropped by orbit or substring: {int((mask_ratio_ok & drop_mask).sum())}")
    print(f"Final rows: {len(df_out)}")
    if out_path:
        print(f"Saved: {out_path}")

    return df_out, out_path

# --- Example call (uncomment and run in a notebook) ---
df_out, out_path = filter_matched_all()
# df_out.head()


Processing Method 1...
Processing Method 2...
Processing Method 3...
Processing Method 4...

✅ All plots saved to: all_methods_plots_paper_large
